# AI-Based Fabric Defect Detection with MVTec AD  
## Portfolio-Ready Notebook

This notebook builds and compares three anomaly-detection approaches for **fabric-like industrial texture inspection** using the **`carpet`** category from **MVTec AD**.

### Goal
Detect whether a fabric image is:
- **normal** (`label = 0`)
- **defective** (`label = 1`)

### Experiment roadmap
1. **Autoencoder baseline** — train on normal images only, detect anomalies using reconstruction error  
2. **Image-level ResNet18 embeddings** — nearest-neighbor anomaly scoring on global image features  
3. **Patch-level ResNet18 embeddings** — nearest-neighbor anomaly scoring on local patch features  

### Final takeaway
The patch-level method is the final selected MVP because defects in industrial textures are often **local**, not global.

## 1. Setup
This section imports packages, fixes randomness where possible, and prints the runtime device.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
)
from sklearn.preprocessing import normalize

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 2. Configuration
Adjust these values if you want to experiment with a different category, image size, or memory-bank size.

In [ ]:
BASE_PATH = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")
CATEGORY = "carpet"

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2

AE_EPOCHS = 10
AE_LR = 1e-3

IMG_KNN_NEIGHBORS = 5
PATCH_KNN_NEIGHBORS = 1
PATCH_TOP_K = 5
MAX_PATCHES = 20000
CALIBRATION_RATIO = 0.20
CALIBRATION_QUANTILE = 0.95

assert BASE_PATH.exists(), f"Dataset path not found: {BASE_PATH}"
print("Dataset root exists:", BASE_PATH)
print("Available folders:", sorted([p.name for p in BASE_PATH.iterdir() if p.is_dir()])[:20])


## 3. Build the `carpet` dataframe
MVTec AD is structured for anomaly detection:
- `train/good` contains **normal images only**
- `test/good` contains normal test images
- `test/<defect_type>` contains anomalous images

We convert this into a dataframe with:
- `image_path`
- `split`
- `label`
- `defect_type`

In [ ]:
def build_mvtec_binary_dataframe(base_path: Path, category: str) -> pd.DataFrame:
    rows = []
    category_path = base_path / category

    for img_path in (category_path / "train" / "good").glob("*.*"):
        rows.append({
            "image_path": str(img_path),
            "split": "train",
            "label": 0,
            "defect_type": "good",
        })

    for defect_folder in (category_path / "test").iterdir():
        if defect_folder.is_dir():
            defect_type = defect_folder.name
            label = 0 if defect_type == "good" else 1
            for img_path in defect_folder.glob("*.*"):
                rows.append({
                    "image_path": str(img_path),
                    "split": "test",
                    "label": label,
                    "defect_type": defect_type,
                })

    return pd.DataFrame(rows)

df = build_mvtec_binary_dataframe(BASE_PATH, CATEGORY)
full_train_df = df[df["split"] == "train"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

train_df, calibration_df = train_test_split(
    full_train_df,
    test_size=CALIBRATION_RATIO,
    random_state=SEED,
    shuffle=True,
)
train_df = train_df.reset_index(drop=True)
calibration_df = calibration_df.reset_index(drop=True)

print("Train fit images:", len(train_df))
print("Calibration normal images:", len(calibration_df))
print("Test images:", len(test_df))
print("\nTest label distribution:")
print(test_df["label"].value_counts())
print("\nTest defect types:")
print(test_df["defect_type"].value_counts())


## 4. Shared utilities
One dataset class is reused across all experiments.  
We also define helper functions for visualization, threshold selection, and reporting.

In [ ]:
class MVTecDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        label = int(row["label"])

        if self.transform:
            image = self.transform(image)

        return image, label, row["image_path"]


imagenet_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

ae_train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

ae_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


def make_loader(df, transform, batch_size=BATCH_SIZE):
    dataset = MVTecDataset(df, transform=transform)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )
    return dataset, loader


def show_image_grid(df_subset, n=6, title="Samples"):
    plt.figure(figsize=(15, 4))
    for i in range(min(n, len(df_subset))):
        row = df_subset.iloc[i]
        img = Image.open(row["image_path"]).convert("RGB")
        plt.subplot(1, n, i + 1)
        plt.imshow(img)
        plt.title(f"{row['defect_type']}\nlabel={row['label']}")
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_confusion(cm, title="Confusion Matrix"):
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["normal", "defect"])
    ax.set_yticklabels(["normal", "defect"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", color="black")

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


def plot_score_histogram(normal_scores, anomaly_scores, title):
    plt.figure(figsize=(8, 5))
    plt.hist(normal_scores, bins=30, alpha=0.6, label="normal")
    plt.hist(anomaly_scores, bins=30, alpha=0.6, label="anomaly")
    plt.legend()
    plt.title(title)
    plt.xlabel("Anomaly score")
    plt.ylabel("Count")
    plt.show()


def calibrate_threshold_from_normals(normal_scores, quantile=CALIBRATION_QUANTILE):
    return float(np.quantile(normal_scores, quantile))


def summarize_results(name, labels, preds, scores, threshold, threshold_strategy):
    cm = confusion_matrix(labels, preds)
    report = classification_report(labels, preds, output_dict=True)
    auc = roc_auc_score(labels, scores)

    return {
        "experiment": name,
        "accuracy": report["accuracy"],
        "precision": report["1"]["precision"],
        "recall": report["1"]["recall"],
        "f1_normal": report["0"]["f1-score"],
        "f1_defect": report["1"]["f1-score"],
        "roc_auc": auc,
        "threshold": float(threshold),
        "threshold_strategy": threshold_strategy,
        "false_positives": int(cm[0, 1]),
        "false_negatives": int(cm[1, 0]),
        "cm": cm,
    }


def show_ranked_images(results_df, title, n=6):
    plt.figure(figsize=(15, 4))
    for i in range(min(n, len(results_df))):
        row = results_df.iloc[i]
        img = Image.open(row["image_path"]).convert("RGB")
        plt.subplot(1, n, i + 1)
        plt.imshow(img)
        plt.title(f"GT:{row['label']} P:{row['pred']}\nS:{row['score']:.3f}")
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


### Quick qualitative check
A few normal training images and a few test images:

In [ ]:
show_image_grid(train_df.sample(6, random_state=SEED).reset_index(drop=True), title="Normal training images")
show_image_grid(test_df.sample(6, random_state=SEED).reset_index(drop=True), title="Mixed test images")

# 5. Experiment 1 — Autoencoder Baseline

### Hypothesis
If the model is trained only on normal images, it should reconstruct normal textures well and reconstruct defects poorly.

### Expected weakness
Subtle texture defects may still be reconstructed too well, which reduces anomaly separability.

In [ ]:
ae_train_dataset, ae_train_loader = make_loader(
    train_df,
    ae_train_transform,
    batch_size=BATCH_SIZE,
)
ae_calibration_dataset, ae_calibration_loader = make_loader(
    calibration_df,
    ae_test_transform,
    batch_size=BATCH_SIZE,
)
ae_test_dataset, ae_test_loader = make_loader(
    test_df,
    ae_test_transform,
    batch_size=BATCH_SIZE,
)

class AutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


ae_model = AutoEncoder().to(device)
ae_optimizer = torch.optim.Adam(ae_model.parameters(), lr=AE_LR)
ae_criterion = nn.MSELoss()


In [ ]:
ae_history = []

for epoch in range(AE_EPOCHS):
    ae_model.train()
    running_loss = 0.0

    for images, _, _ in ae_train_loader:
        images = images.to(device)

        outputs = ae_model(images)
        loss = ae_criterion(outputs, images)

        ae_optimizer.zero_grad()
        loss.backward()
        ae_optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(ae_train_loader)
    ae_history.append(epoch_loss)
    print(f"Epoch {epoch + 1}/{AE_EPOCHS} - loss: {epoch_loss:.4f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(ae_history)
plt.title("Autoencoder Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.show()

In [ ]:
ae_model.eval()
ae_calibration_errors = []

with torch.no_grad():
    for images, _, _ in ae_calibration_loader:
        images = images.to(device)
        outputs = ae_model(images)
        batch_errors = torch.mean((images - outputs) ** 2, dim=[1, 2, 3])
        ae_calibration_errors.extend(batch_errors.cpu().numpy())

ae_threshold = calibrate_threshold_from_normals(
    np.array(ae_calibration_errors),
    quantile=CALIBRATION_QUANTILE,
)

ae_errors = []
ae_labels = []
ae_paths = []

with torch.no_grad():
    for images, labels, paths in ae_test_loader:
        images = images.to(device)
        outputs = ae_model(images)

        batch_errors = torch.mean((images - outputs) ** 2, dim=[1, 2, 3])

        ae_errors.extend(batch_errors.cpu().numpy())
        ae_labels.extend(labels.numpy())
        ae_paths.extend(paths)

ae_errors = np.array(ae_errors)
ae_labels = np.array(ae_labels)

ae_preds = (ae_errors > ae_threshold).astype(int)

ae_threshold_strategy = f"normal_quantile_{CALIBRATION_QUANTILE:.2f}"
print("Calibration threshold:", ae_threshold)
print("Threshold strategy:", ae_threshold_strategy)
print(classification_report(ae_labels, ae_preds))
ae_cm = confusion_matrix(ae_labels, ae_preds)
print(ae_cm)
print("ROC-AUC:", roc_auc_score(ae_labels, ae_errors))
plot_confusion(ae_cm, title="Autoencoder Baseline - Confusion Matrix")
plot_score_histogram(ae_errors[ae_labels == 0], ae_errors[ae_labels == 1], "Autoencoder Reconstruction Error Distribution")

ae_results_df = pd.DataFrame({
    "image_path": ae_paths,
    "label": ae_labels,
    "score": ae_errors,
    "pred": ae_preds,
}).sort_values("score", ascending=False).reset_index(drop=True)

show_ranked_images(ae_results_df[ae_results_df["label"] == 1], title="Autoencoder - Top defect scores")
show_ranked_images(ae_results_df[ae_results_df["label"] == 0].sort_values("score", ascending=False), title="Autoencoder - Highest-scoring normals")

autoencoder_summary = summarize_results(
    "Autoencoder baseline",
    ae_labels,
    ae_preds,
    ae_errors,
    threshold=ae_threshold,
    threshold_strategy=ae_threshold_strategy,
)


## Experiment 1 conclusion
The plain convolutional autoencoder is a weak baseline for this problem.  
It tends to reconstruct some defective textures too well, which compresses the gap between normal and anomalous reconstruction errors.

# 6. Experiment 2 — Image-Level ResNet18 Embeddings

### Idea
Use a pretrained ResNet18 as a frozen feature extractor:
- extract one embedding per image
- build a normal-image memory bank
- score each test image using nearest-neighbor distance to normal training features

### Why this should improve
Pretrained convolutional features capture richer visual structure than pixel-level reconstruction error.

In [ ]:
img_train_dataset, img_train_loader = make_loader(
    train_df,
    imagenet_transform,
    batch_size=32,
)
img_calibration_dataset, img_calibration_loader = make_loader(
    calibration_df,
    imagenet_transform,
    batch_size=32,
)
img_test_dataset, img_test_loader = make_loader(
    test_df,
    imagenet_transform,
    batch_size=32,
)

resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
image_feature_extractor = nn.Sequential(*list(resnet.children())[:-1]).to(device)
image_feature_extractor.eval()

for p in image_feature_extractor.parameters():
    p.requires_grad = False


def extract_image_features(loader, model, device):
    features, labels, paths = [], [], []

    model.eval()
    with torch.no_grad():
        for images, batch_labels, batch_paths in loader:
            images = images.to(device)
            outputs = model(images).view(images.size(0), -1)

            features.append(outputs.cpu().numpy())
            labels.extend(batch_labels.numpy())
            paths.extend(batch_paths)

    features = np.vstack(features)
    labels = np.array(labels)
    return features, labels, paths


img_train_features, img_train_labels, img_train_paths = extract_image_features(
    img_train_loader, image_feature_extractor, device
)
img_calibration_features, img_calibration_labels, img_calibration_paths = extract_image_features(
    img_calibration_loader, image_feature_extractor, device
)
img_test_features, img_test_labels, img_test_paths = extract_image_features(
    img_test_loader, image_feature_extractor, device
)

img_train_features = normalize(img_train_features)
img_calibration_features = normalize(img_calibration_features)
img_test_features = normalize(img_test_features)

print("Train features shape:", img_train_features.shape)
print("Calibration features shape:", img_calibration_features.shape)
print("Test features shape:", img_test_features.shape)


In [ ]:
img_nn = NearestNeighbors(n_neighbors=IMG_KNN_NEIGHBORS, metric="cosine")
img_nn.fit(img_train_features)

img_calibration_distances, _ = img_nn.kneighbors(img_calibration_features)
img_calibration_scores = img_calibration_distances.mean(axis=1)
img_threshold = calibrate_threshold_from_normals(
    img_calibration_scores,
    quantile=CALIBRATION_QUANTILE,
)
img_threshold_strategy = f"normal_quantile_{CALIBRATION_QUANTILE:.2f}"

img_distances, _ = img_nn.kneighbors(img_test_features)
img_scores = img_distances.mean(axis=1)
img_preds = (img_scores > img_threshold).astype(int)

print("Calibration threshold:", img_threshold)
print("Threshold strategy:", img_threshold_strategy)
print(classification_report(img_test_labels, img_preds))
img_cm = confusion_matrix(img_test_labels, img_preds)
print(img_cm)
print("ROC-AUC:", roc_auc_score(img_test_labels, img_scores))

plot_confusion(img_cm, title="Image-Level Embeddings - Confusion Matrix")
plot_score_histogram(
    img_scores[img_test_labels == 0],
    img_scores[img_test_labels == 1],
    "Image-Level Nearest-Neighbor Distance Distribution",
)

img_results_df = pd.DataFrame({
    "image_path": img_test_paths,
    "label": img_test_labels,
    "score": img_scores,
    "pred": img_preds,
}).sort_values("score", ascending=False).reset_index(drop=True)

show_ranked_images(img_results_df[img_results_df["label"] == 1], title="Image-level method - Top defect scores")
show_ranked_images(img_results_df[img_results_df["label"] == 0].sort_values("score", ascending=False), title="Image-level method - Highest-scoring normals")

image_level_summary = summarize_results(
    "Image-level ResNet18 + kNN",
    img_test_labels,
    img_preds,
    img_scores,
    threshold=img_threshold,
    threshold_strategy=img_threshold_strategy,
)


## Experiment 2 conclusion
Image-level feature embeddings improve anomaly detection substantially compared with the autoencoder baseline.  
However, a single global descriptor can still miss or blur small local defects.

# 7. Experiment 3 — Patch-Level ResNet18 Embeddings

### Idea
Extract an intermediate feature map from ResNet18 and treat each spatial location as a local patch embedding.

For each test image:
- compute the nearest normal patch distance for every patch
- use the average of the top-`k` highest patch distances as the image anomaly score

### Why this should help
Industrial texture defects are often **small, localized regions**.  
Patch-level scoring preserves this local anomaly signal and also enables heatmap visualization.

In [ ]:
patch_train_dataset, patch_train_loader = make_loader(
    train_df,
    imagenet_transform,
    batch_size=16,
)
patch_calibration_dataset, patch_calibration_loader = make_loader(
    calibration_df,
    imagenet_transform,
    batch_size=16,
)
patch_test_dataset, patch_test_loader = make_loader(
    test_df,
    imagenet_transform,
    batch_size=16,
)

patch_backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT).to(device)
patch_backbone.eval()

for p in patch_backbone.parameters():
    p.requires_grad = False


class ResNetPatchExtractor(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return x


patch_extractor = ResNetPatchExtractor(patch_backbone).to(device)
patch_extractor.eval()


In [ ]:
def extract_patch_embeddings(loader, model, device):
    all_image_patch_embeddings = []
    all_labels = []
    all_paths = []

    model.eval()
    with torch.no_grad():
        for images, labels, paths in loader:
            images = images.to(device)
            feat_maps = model(images)

            B, C, H, W = feat_maps.shape
            patches = feat_maps.permute(0, 2, 3, 1).contiguous().view(B, H * W, C)
            patches = F.normalize(patches, p=2, dim=-1)

            for i in range(B):
                all_image_patch_embeddings.append(patches[i].cpu())
                all_labels.append(labels[i].item())
                all_paths.append(paths[i])

    return all_image_patch_embeddings, np.array(all_labels), all_paths


train_patch_list, train_patch_labels, train_patch_paths = extract_patch_embeddings(
    patch_train_loader, patch_extractor, device
)
calibration_patch_list, calibration_patch_labels, calibration_patch_paths = extract_patch_embeddings(
    patch_calibration_loader, patch_extractor, device
)
test_patch_list, test_patch_labels, test_patch_paths = extract_patch_embeddings(
    patch_test_loader, patch_extractor, device
)

print("Number of train images:", len(train_patch_list))
print("Number of calibration images:", len(calibration_patch_list))
print("Number of test images:", len(test_patch_list))
print("Patch shape for one image:", train_patch_list[0].shape)


In [ ]:
train_patch_bank = torch.cat(train_patch_list, dim=0).reshape(-1, train_patch_list[0].shape[-1]).numpy()
print("Full patch bank shape:", train_patch_bank.shape)

if len(train_patch_bank) > MAX_PATCHES:
    sample_idx = np.random.choice(len(train_patch_bank), MAX_PATCHES, replace=False)
    train_patch_bank = train_patch_bank[sample_idx]

print("Memory bank used shape:", train_patch_bank.shape)

patch_nn = NearestNeighbors(n_neighbors=PATCH_KNN_NEIGHBORS, metric="cosine")
patch_nn.fit(train_patch_bank)


In [ ]:
def compute_patch_anomaly_scores(test_patch_list, nn_model, top_k=PATCH_TOP_K):
    image_scores = []
    patch_score_maps = []

    for patches in test_patch_list:
        patches_np = patches.numpy()
        distances, _ = nn_model.kneighbors(patches_np)
        patch_scores = distances.squeeze()

        top_scores = np.sort(patch_scores)[-top_k:]
        image_score = top_scores.mean()

        image_scores.append(image_score)
        patch_score_maps.append(patch_scores)

    return np.array(image_scores), patch_score_maps


calibration_patch_scores, _ = compute_patch_anomaly_scores(
    calibration_patch_list,
    patch_nn,
    top_k=PATCH_TOP_K,
)
patch_threshold = calibrate_threshold_from_normals(
    calibration_patch_scores,
    quantile=CALIBRATION_QUANTILE,
)
patch_threshold_strategy = f"normal_quantile_{CALIBRATION_QUANTILE:.2f}"

patch_scores, patch_score_maps = compute_patch_anomaly_scores(
    test_patch_list, patch_nn, top_k=PATCH_TOP_K
)
patch_preds = (patch_scores > patch_threshold).astype(int)

print("Calibration threshold:", patch_threshold)
print("Threshold strategy:", patch_threshold_strategy)
print(classification_report(test_patch_labels, patch_preds))
patch_cm = confusion_matrix(test_patch_labels, patch_preds)
print(patch_cm)
print("ROC-AUC:", roc_auc_score(test_patch_labels, patch_scores))

plot_confusion(patch_cm, title="Patch-Level Embeddings - Confusion Matrix")
plot_score_histogram(
    patch_scores[test_patch_labels == 0],
    patch_scores[test_patch_labels == 1],
    "Patch-Level Anomaly Score Distribution",
)

patch_results_df = pd.DataFrame({
    "image_path": test_patch_paths,
    "label": test_patch_labels,
    "score": patch_scores,
    "pred": patch_preds,
}).sort_values("score", ascending=False).reset_index(drop=True)

show_ranked_images(patch_results_df[patch_results_df["label"] == 1], title="Patch-level method - Top defect scores")
show_ranked_images(patch_results_df[patch_results_df["label"] == 0].sort_values("score", ascending=False), title="Patch-level method - Highest-scoring normals")

patch_level_summary = summarize_results(
    "Patch-level ResNet18 + kNN",
    test_patch_labels,
    patch_preds,
    patch_scores,
    threshold=patch_threshold,
    threshold_strategy=patch_threshold_strategy,
)


In [ ]:
def show_patch_heatmap(image_path, patch_scores, img_size=IMG_SIZE, feature_map_hw=None):
    image = Image.open(image_path).convert("RGB").resize((img_size, img_size))
    image_np = np.array(image)

    if feature_map_hw is None:
        num_patches = len(patch_scores)
        side = int(np.sqrt(num_patches))
        feature_map_hw = (side, side)

    heatmap = patch_scores.reshape(feature_map_hw)
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)

    heatmap_tensor = torch.tensor(heatmap).unsqueeze(0).unsqueeze(0).float()
    heatmap_up = F.interpolate(
        heatmap_tensor,
        size=(img_size, img_size),
        mode="bilinear",
        align_corners=False,
    ).squeeze().numpy()

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(image_np)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(heatmap_up, cmap="jet")
    plt.title("Patch Anomaly Heatmap")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(image_np)
    plt.imshow(heatmap_up, cmap="jet", alpha=0.45)
    plt.title("Overlay")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Visualize one high-scoring defect heatmap
defect_indices = np.where(test_patch_labels == 1)[0]
sorted_defects = defect_indices[np.argsort(patch_scores[defect_indices])[::-1]]

sample_idx = sorted_defects[0]
sample_path = test_patch_paths[sample_idx]
sample_patch_scores = patch_score_maps[sample_idx]

num_patches = len(sample_patch_scores)
side = int(np.sqrt(num_patches))

show_patch_heatmap(sample_path, sample_patch_scores, feature_map_hw=(side, side))

## Experiment 3 conclusion
Patch-level nearest-neighbor anomaly detection is the strongest method in this notebook.  
It improves both discrimination and localization because it preserves **local defect evidence** that is lost in whole-image descriptors.

# 8. Final comparison
The table below compares the three experiments on the same `carpet` test split.

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Experiment": autoencoder_summary["experiment"],
        "Accuracy": autoencoder_summary["accuracy"],
        "Precision": autoencoder_summary["precision"],
        "Recall": autoencoder_summary["recall"],
        "F1 (normal)": autoencoder_summary["f1_normal"],
        "F1 (defect)": autoencoder_summary["f1_defect"],
        "ROC-AUC": autoencoder_summary["roc_auc"],
        "Threshold": autoencoder_summary["threshold"],
        "Threshold strategy": autoencoder_summary["threshold_strategy"],
        "False positives": autoencoder_summary["false_positives"],
        "False negatives": autoencoder_summary["false_negatives"],
    },
    {
        "Experiment": image_level_summary["experiment"],
        "Accuracy": image_level_summary["accuracy"],
        "Precision": image_level_summary["precision"],
        "Recall": image_level_summary["recall"],
        "F1 (normal)": image_level_summary["f1_normal"],
        "F1 (defect)": image_level_summary["f1_defect"],
        "ROC-AUC": image_level_summary["roc_auc"],
        "Threshold": image_level_summary["threshold"],
        "Threshold strategy": image_level_summary["threshold_strategy"],
        "False positives": image_level_summary["false_positives"],
        "False negatives": image_level_summary["false_negatives"],
    },
    {
        "Experiment": patch_level_summary["experiment"],
        "Accuracy": patch_level_summary["accuracy"],
        "Precision": patch_level_summary["precision"],
        "Recall": patch_level_summary["recall"],
        "F1 (normal)": patch_level_summary["f1_normal"],
        "F1 (defect)": patch_level_summary["f1_defect"],
        "ROC-AUC": patch_level_summary["roc_auc"],
        "Threshold": patch_level_summary["threshold"],
        "Threshold strategy": patch_level_summary["threshold_strategy"],
        "False positives": patch_level_summary["false_positives"],
        "False negatives": patch_level_summary["false_negatives"],
    },
])

comparison_df = comparison_df.sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
comparison_df


# 9. Final conclusion

### Selected MVP
**Patch-level ResNet18 embeddings + nearest-neighbor anomaly scoring**

### Why it wins
- highest anomaly discrimination
- much better balance between missed defects and false alarms
- supports heatmap-based localization
- more faithful to the local nature of industrial texture defects

### Honest methodology note
The image-level and patch-level thresholds are now calibrated on a **held-out normal calibration split** carved from the training data.
That keeps the test set reserved for final evaluation and matches the deployment story used in the main project repo.


# 10. Limitations and future work

## Limitations
- This notebook evaluates the **`carpet`** category only.
- Threshold calibration uses a held-out normal subset, so it does not directly optimize defect recall.
- The memory bank is randomly subsampled for computational efficiency.
- This is a nearest-neighbor baseline, not a full implementation of PatchCore or PaDiM.

## Future work
- extend the pipeline to `grid` and `leather`
- compare calibration by normal quantile versus validation F1 on a labeled validation split
- compare against PatchCore, PaDiM, and FastFlow
- package the final model into a FastAPI service
- build a Streamlit demo with image upload and anomaly heatmap output


# 11. Portfolio-ready project summary

You can reuse this wording in your README or portfolio:

> Built an industrial anomaly detection system for fabric-like texture inspection using MVTec AD.
> Compared three approaches?autoencoder reconstruction, image-level ResNet18 embeddings, and patch-level ResNet18 embeddings?and selected patch-level nearest-neighbor anomaly detection as the final MVP because it achieved the best trade-off between defect detection performance and local defect localization.
> Calibrated anomaly thresholds on a held-out normal calibration split so the final test set stayed untouched until evaluation.

### Resume bullet
- Developed a patch-level anomaly detection pipeline for industrial texture inspection using ResNet18 feature maps, nearest-neighbor scoring, and normal-only threshold calibration, achieving strong defect detection performance with interpretable heatmap localization on MVTec AD.
